# ROS1 Kinase Variant MD Analysis — Results Notebook
**Lily Konadu Gyammerah | Hanze University | Data Science for Life Sciences**

This notebook reproduces and visualises all key results from the ROS1 MD analysis pipeline.
Sections follow the thesis chapter order: PCA/FEL → Clustering → Active Site Metrics → LDA → Docking.

**How to use:** Set `BASE_DIR` in Section 0 to your results directory, then run cells in order.
Each section is self-contained — you can re-run individual sections without re-running everything.

---
| Section | Content |
|---------|--------|
| 0 | Setup & paths |
| 1 | PCA overview — explained variance & stride verification |
| 2 | Free energy landscapes — browse all 39 systems |
| 3 | DBSCAN noise fractions — panel overview |
| 4 | Active site structural metrics |
| 5 | Distance PCA (NTL–CTL inter-lobe) |
| 6 | LDA — Q2022P family vs WT |
| 7 | Same-position contrast pairs |
| 8 | Grey highlight plots — all 39 systems |
| 9 | CTL FEL overview |
| 10 | Docking results (Q2022P family) |
| 11 | Export summary table |


## Section 0 — Setup & Paths

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import os, glob, warnings
warnings.filterwarnings('ignore')

# ── Adjust this path to your results directory on the cluster ──────────────
BASE_DIR = os.path.expanduser(
    '~/Molecular_Dynamics_analysis/results/ROS1/ros1_prepared_final'
)
FIG_DIR  = os.path.join(BASE_DIR, 'figures')

# Ordered variant list (index 0..38 matches x_owner integer values)
LABELS = [
    'WT','C2060G','D1988N','D2033N','D2113G','D2113N','E1935G','E1990G',
    'E2020K','E2131Q','F1994L','F2004C','F2004L','F2004V','F2075V',
    'G1957A','G1971E','G2032K','G2032R','G2048A_NC','G2101A','H1999Q_NC',
    'L1947R','L1951R','L1982F','L1982V','L2010M','L2026M','L2053V_NC',
    'L2086F','L2155S','Q2022P','Q2022P_S1986F','Q2022P_S1986Y',
    'R2078K','S1986F','S1986Y','V2089M','V2098I'
]
N_SYS = len(LABELS)
WT_IDX = LABELS.index('WT')

# Colour palette for key mutants
PALETTE = {
    'WT':'#2196F3', 'Q2022P':'#E63946', 'Q2022P_S1986F':'#FF6B35',
    'Q2022P_S1986Y':'#FFBE0B', 'S1986F':'#4CAF50', 'S1986Y':'#8BC34A',
    'G2032K':'#9C27B0', 'G2032R':'#673AB7', 'L1982F':'#F06292',
    'L1982V':'#EC407A', 'F2004C':'#00BCD4', 'F2004L':'#0097A7',
    'F2004V':'#26C6DA', 'D2113G':'#FF5722', 'D2113N':'#FF7043',
}
DEFAULT_COLOUR = '#888888'

# Custom FEL colourmap matching the thesis (viridis-based, 0–2.5 kT)
FEL_CMAP = plt.get_cmap('viridis')
FEL_VMAX = 2.5

print(f'BASE_DIR: {BASE_DIR}')
print(f'Variants: {N_SYS}')
print('Setup complete.')


In [ ]:
# Load core arrays
scores   = np.load(os.path.join(BASE_DIR, 'actout_ctlfit_scores.npy'))   # (N, n_pcs)
owner    = np.load(os.path.join(BASE_DIR, 'x_owner.npy'))                 # (N,)
print(f'scores shape : {scores.shape}')
print(f'owner shape  : {owner.shape}')
print(f'frames per system: min={np.bincount(owner).min()}, max={np.bincount(owner).max()}')


## Section 1 — PCA Overview: Explained Variance & Stride Verification

In [ ]:
# Load explained variance if available, else compute from scores covariance
ev_file = os.path.join(BASE_DIR, 'actout_ctlfit_explained_variance.npy')
if os.path.exists(ev_file):
    ev = np.load(ev_file)
else:
    # Approximate from score variances (not exact but good enough for visualisation)
    ev = np.var(scores, axis=0)
    ev = ev / ev.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
n_show = min(20, len(ev))
axes[0].bar(range(1, n_show+1), ev[:n_show], color='#2E4057', alpha=0.85)
axes[0].set_xlabel('Principal Component', fontsize=11)
axes[0].set_ylabel('Explained Variance (%)', fontsize=11)
axes[0].set_title('actout_ctlfit PCA — Scree Plot', fontsize=12, fontweight='bold')
axes[0].axvline(2.5, color='#E63946', lw=1.5, ls='--', label='PC1+PC2 cutoff')
axes[0].legend(fontsize=9)

# Cumulative explained variance
cum_ev = np.cumsum(ev[:n_show])
axes[1].plot(range(1, n_show+1), cum_ev, 'o-', color='#2E4057', lw=2)
axes[1].axhline(50, color='#E63946', lw=1.5, ls='--', label='50% threshold')
axes[1].set_xlabel('Number of PCs', fontsize=11)
axes[1].set_ylabel('Cumulative Explained Variance (%)', fontsize=11)
axes[1].set_title('Cumulative Explained Variance', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'pca_scree_cumulative.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'PC1+PC2 explain: {ev[0]+ev[1]:.1f}%')


In [ ]:
# Stride verification summary table (from confirmed CSV values)
stride_data = {
    'PCA Space': ['actout_ctlfit']*5 + ['aloop']*5,
    'PC': ['PC1','PC2','PC3','PC4','PC5'] * 2,
    'Stride 1 vs 10': [1.000, 0.9985, 0.9982, 0.9930, 0.9910,
                        1.000, 0.9993, 0.9980, 0.9930, 0.9860],
    'Stride 5 vs 10': [1.000, 0.9989, 0.9984, 0.9980, 0.9970,
                        1.000, 0.9995, 0.9990, 0.9810, 0.9790],
    'Pass (≥0.98)': ['✅']*10
}
df_stride = pd.DataFrame(stride_data)
print('Stride Verification Results (WT replica 0, traj_index=111)')
print('=' * 65)
print(df_stride.to_string(index=False))
print('\nConclusion: All primary PCA modes pass stride-10 stability check.')


## Section 2 — Free Energy Landscapes: Browse All 39 Systems

In [ ]:
def compute_fel(pc1_frames, pc2_frames, bins=120, vmax=2.5, kT=1.0):
    """Compute 2D free energy landscape from PCA projections."""
    h, xedges, yedges = np.histogram2d(pc1_frames, pc2_frames, bins=bins)
    h = h.T  # transpose so rows = y
    with np.errstate(divide='ignore'):
        G = -kT * np.log(h / h.max())
    G[np.isinf(G)] = np.nan
    G = np.clip(G, 0, vmax)
    xc = 0.5 * (xedges[:-1] + xedges[1:])
    yc = 0.5 * (yedges[:-1] + yedges[1:])
    return G, xc, yc

def plot_fel(ax, pc1_frames, pc2_frames, title, xlim=None, ylim=None,
             bins=120, vmax=2.5, cmap='viridis_r'):
    """Plot FEL on given axes."""
    G, xc, yc = compute_fel(pc1_frames, pc2_frames, bins=bins, vmax=vmax)
    XX, YY = np.meshgrid(xc, yc)
    im = ax.pcolormesh(XX, YY, G, cmap=cmap, vmin=0, vmax=vmax,
                       shading='auto', rasterized=True)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('PC1', fontsize=8)
    ax.set_ylabel('PC2', fontsize=8)
    ax.tick_params(labelsize=7)
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)
    return im

# Fixed axis limits from all-system percentiles
pc1_all, pc2_all = scores[:,0], scores[:,1]
XLIM = (np.percentile(pc1_all,0.5), np.percentile(pc1_all,99.5))
YLIM = (np.percentile(pc2_all,0.5), np.percentile(pc2_all,99.5))

print(f'Fixed axis limits:  PC1 {XLIM[0]:.3f} → {XLIM[1]:.3f}')
print(f'                    PC2 {YLIM[0]:.3f} → {YLIM[1]:.3f}')
print('FEL helper functions defined.')


In [ ]:
# ── 6×7 panel of all 39 FELs ──────────────────────────────────────────────────
n_cols, n_rows = 6, 7
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3, n_rows*3), dpi=100)
axes = axes.flatten()

im_last = None
for i, name in enumerate(LABELS):
    mask = owner == i
    if mask.sum() < 10:
        axes[i].axis('off')
        continue
    im_last = plot_fel(axes[i], pc1_all[mask], pc2_all[mask],
                        title=name, xlim=XLIM, ylim=YLIM)

# Turn off unused subplots
for j in range(N_SYS, len(axes)):
    axes[j].axis('off')

# Shared colourbar
fig.subplots_adjust(right=0.92, hspace=0.45, wspace=0.35)
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
cb = fig.colorbar(im_last, cax=cbar_ax)
cb.set_label('Relative free energy (kT)', fontsize=10)

fig.suptitle('actout_ctlfit Free Energy Landscapes — All 39 Systems',
             fontsize=13, fontweight='bold', y=1.01)

os.makedirs(FIG_DIR, exist_ok=True)
fig.savefig(os.path.join(FIG_DIR, 'FEL_panel_all_actout.png'),
            dpi=120, bbox_inches='tight', facecolor='white')
plt.show()
print('FEL panel saved.')


In [ ]:
# ── Interactive single-mutant FEL browser ─────────────────────────────────────
# Change FOCAL_MUTANT to any name in LABELS to inspect that mutant
FOCAL_MUTANT = 'Q2022P'   # ← change this

idx = LABELS.index(FOCAL_MUTANT)
mask = owner == idx
wt_mask = owner == WT_IDX

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_fel(axes[0], pc1_all[wt_mask], pc2_all[wt_mask],
         title='WT (reference)', xlim=XLIM, ylim=YLIM)
im = plot_fel(axes[1], pc1_all[mask], pc2_all[mask],
               title=f'{FOCAL_MUTANT}  (n={mask.sum():,} frames)',
               xlim=XLIM, ylim=YLIM)
fig.colorbar(im, ax=axes[1], label='Relative free energy (kT)')
plt.suptitle(f'FEL Comparison: WT vs {FOCAL_MUTANT}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 3 — DBSCAN Noise Fractions: Panel Overview

In [ ]:
# Confirmed noise fractions from pipeline (all 37 mutants + WT)
noise_data = {
    'Variant': ['L1982V','G2048A_NC','L1951R','F2004C','Q2022P','S1986Y','E2131Q',
                'V2098I','L2086F','E2020K','L2155S','Q2022P_S1986Y','L1947R',
                'H1999Q_NC','Q2022P_S1986F','L1982F','C2060G','F2004L','L2053V_NC',
                'D1988N','V2089M','R2078K','G2101A','WT','E1990G','L2010M',
                'D2033N','G2032R','F2075V','F2004V','E1935G','G1957A','S1986F',
                'G2032K','G1971E','D2113G','D2113N'],
    'Noise_Fraction_%': [7.27,5.21,5.18,5.14,4.51,4.31,4.28,
                          3.61,3.50,2.50,2.26,2.07,1.95,
                          1.64,1.53,1.45,1.27,1.26,1.36,
                          1.24,1.20,0.95,0.84,0.83,0.83,0.82,
                          0.79,0.68,0.55,0.44,0.34,0.30,0.25,
                          0.21,0.10,0.16,0.08]
}
df_noise = pd.DataFrame(noise_data).sort_values('Noise_Fraction_%', ascending=False)

# Colour bars: red = NC variants, blue = WT, orange = Q2022P family, grey = others
def bar_colour(name):
    if name in ['G2048A_NC','H1999Q_NC','L2053V_NC']: return '#607D8B'
    if name == 'WT': return '#2196F3'
    if 'Q2022P' in name: return '#E63946'
    return '#78909C'

colours = [bar_colour(n) for n in df_noise['Variant']]

fig, ax = plt.subplots(figsize=(14, 5), dpi=120)
bars = ax.bar(range(len(df_noise)), df_noise['Noise_Fraction_%'],
               color=colours, edgecolor='white', linewidth=0.4)

ax.set_xticks(range(len(df_noise)))
ax.set_xticklabels(df_noise['Variant'], rotation=60, ha='right', fontsize=7.5)
ax.set_ylabel('DBSCAN Noise Fraction (%)', fontsize=11)
ax.set_title('Per-Mutant DBSCAN Noise Fraction — actout_ctlfit Space',
             fontsize=12, fontweight='bold')
ax.axhline(0.83, color='#2196F3', lw=1.5, ls='--', label=f'WT (0.83%)')

# Custom legend
from matplotlib.patches import Patch
legend_els = [
    Patch(fc='#2196F3', label='WT'),
    Patch(fc='#E63946', label='Q2022P family'),
    Patch(fc='#607D8B', label='Negative controls'),
    Patch(fc='#78909C', label='Other resistance mutants'),
]
ax.legend(handles=legend_els, fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'dbscan_noise_fractions.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(df_noise[['Variant','Noise_Fraction_%']].to_string(index=False))


## Section 4 — Active Site Structural Metrics

In [ ]:
# Load per-mutant structural metrics CSV (output of 03a_metrics_states.py)
metrics_file = os.path.join(BASE_DIR, 'structural_metrics_summary.csv')

if os.path.exists(metrics_file):
    df_metrics = pd.read_csv(metrics_file)
    print(df_metrics.head())
    print(f'\nShape: {df_metrics.shape}')
    print(f'Columns: {list(df_metrics.columns)}')
else:
    print(f'Metrics file not found at:\n  {metrics_file}')
    print('Using confirmed summary values from thesis for illustration...')
    # Key values from thesis confirmed analysis
    df_metrics = pd.DataFrame({
        'Variant': ['WT','Q2022P','Q2022P_S1986F','Q2022P_S1986Y',
                    'D2113N','E2020K','G2048A_NC','G2032K','G2032R',
                    'S1986F','S1986Y'],
        'Pocket_Open_Pct': [84, 49, 51, 50, 27, 88, 89, 89, 88, 82, 79],
        'DFG_in_Pct': [78, 65, 71, 70, 58, 80, 75, 39, 46, 80, 77],
        'aC_hinge_dist_nm': [2.94, 2.73, 2.73, 2.73, 2.61, 2.95, 2.96,
                              2.94, 2.94, 2.85, 2.88],
    })
    print(df_metrics)


In [ ]:
# If full metrics available, plot pocket openness across all mutants
if 'Pocket_Open_Pct' in df_metrics.columns and len(df_metrics) > 5:
    df_sorted = df_metrics.sort_values('Pocket_Open_Pct')
    colours_m = [PALETTE.get(n, DEFAULT_COLOUR) for n in df_sorted['Variant']]

    fig, ax = plt.subplots(figsize=(14, 5), dpi=120)
    ax.barh(range(len(df_sorted)), df_sorted['Pocket_Open_Pct'],
             color=colours_m, edgecolor='white', linewidth=0.4)
    ax.set_yticks(range(len(df_sorted)))
    ax.set_yticklabels(df_sorted['Variant'], fontsize=8)
    ax.axvline(84, color='#2196F3', lw=1.5, ls='--', label='WT (84%)')
    ax.axvline(49, color='#E63946', lw=1.5, ls='--', label='Q2022P (49%)')
    ax.set_xlabel('ATP Pocket Open-State Occupancy (%)', fontsize=11)
    ax.set_title('Pocket Openness Across All Variants', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'pocket_openness_all.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Full metrics CSV not yet available — run 03a_metrics_states.py first.')


## Section 5 — Distance PCA (NTL–CTL Inter-Lobe Geometry)

In [ ]:
dist_scores_file = os.path.join(BASE_DIR, 'dist_pca_scores.npy')
dist_ev_file     = os.path.join(BASE_DIR, 'dist_pca_explained_variance.npy')

if os.path.exists(dist_scores_file):
    dist_scores = np.load(dist_scores_file)
    dist_ev = np.load(dist_ev_file) if os.path.exists(dist_ev_file) else None

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=110)

    # Scree
    if dist_ev is not None:
        axes[0].bar(range(1, 21), dist_ev[:20], color='#2E4057', alpha=0.85)
        axes[0].set_title('Distance PCA Scree', fontsize=11, fontweight='bold')
        axes[0].set_xlabel('PC'); axes[0].set_ylabel('Explained Variance (%)')
        axes[0].set_title(f'PC1 = {dist_ev[0]:.1f}%, PC1+PC2 = {dist_ev[0]+dist_ev[1]:.1f}%',
                           fontsize=9)

    # Per-mutant PC1 mean positions
    pc1_means = np.array([dist_scores[owner==i, 0].mean() for i in range(N_SYS)])
    order = np.argsort(pc1_means)
    colours_dist = [PALETTE.get(LABELS[i], DEFAULT_COLOUR) for i in order]
    axes[1].barh(range(N_SYS), pc1_means[order], color=colours_dist,
                  edgecolor='white', linewidth=0.3)
    axes[1].set_yticks(range(N_SYS))
    axes[1].set_yticklabels([LABELS[i] for i in order], fontsize=7)
    axes[1].axvline(pc1_means[WT_IDX], color='#2196F3', lw=1.5, ls='--', label='WT mean')
    axes[1].set_xlabel('Mean PC1 (distance PCA)', fontsize=9)
    axes[1].set_title('Per-Mutant Distance PCA PC1 Mean', fontsize=10, fontweight='bold')
    axes[1].legend(fontsize=8)

    # Scatter PC1 vs PC2 coloured by system
    for i, name in enumerate(LABELS):
        mask = owner == i
        c = PALETTE.get(name, DEFAULT_COLOUR)
        alpha = 0.7 if name in PALETTE else 0.2
        axes[2].scatter(dist_scores[mask,0], dist_scores[mask,1],
                         c=c, s=0.5, alpha=alpha, rasterized=True)
    axes[2].set_xlabel('PC1 (distance PCA)', fontsize=9)
    axes[2].set_ylabel('PC2 (distance PCA)', fontsize=9)
    axes[2].set_title('Distance PCA Scatter (highlighted variants)', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'distance_pca_overview.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Distance PCA scores not found at:\n  {dist_scores_file}')
    print('Run step 3b distance PCA script first.')


## Section 6 — LDA: Q2022P Family vs WT

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Family members for LDA
FAMILY = ['WT', 'Q2022P', 'Q2022P_S1986F', 'Q2022P_S1986Y']
fam_indices = [LABELS.index(n) for n in FAMILY]

# Select frames belonging to the family
fam_mask = np.isin(owner, fam_indices)
X_fam = scores[fam_mask, :5]   # use first 5 PCs as features
y_fam = owner[fam_mask]

# Map owner indices to family labels
label_map = {idx: name for idx, name in zip(fam_indices, FAMILY)}
y_names = np.array([label_map[yi] for yi in y_fam])

# Fit LDA (WT vs all three mutants)
lda = LinearDiscriminantAnalysis()
ld_scores = lda.fit_transform(X_fam, y_names)

print('LDA explained variance ratio:', lda.explained_variance_ratio_)

# Per-replica trajectory means — compute mean LD1 per replica
# (This requires trajectory index info — approximate here by splitting each system 3 ways)
colours_fam = {'WT':'#2196F3', 'Q2022P':'#E63946',
                'Q2022P_S1986F':'#FF6B35', 'Q2022P_S1986Y':'#FFBE0B'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=120)

# Frame-level distributions
for name in FAMILY:
    mask_f = y_names == name
    axes[0].hist(ld_scores[mask_f, 0], bins=80, alpha=0.55,
                  color=colours_fam[name], label=name, density=True)
axes[0].set_xlabel('LD1 (frame-level projection)', fontsize=10)
axes[0].set_ylabel('Density', fontsize=10)
axes[0].set_title('LDA Frame-Level: Q2022P Family vs WT', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)

# Trajectory means (approximate — split each system's frames into 3 equal batches as replicas)
means_plot = []
for name in FAMILY:
    mask_f = y_names == name
    ld_sys = ld_scores[mask_f, 0]
    n = len(ld_sys) // 3
    for rep in range(3):
        chunk = ld_sys[rep*n:(rep+1)*n]
        means_plot.append({'name': name, 'rep': rep, 'mean_ld1': chunk.mean()})

df_means = pd.DataFrame(means_plot)
for j, name in enumerate(FAMILY):
    sub = df_means[df_means['name']==name]
    axes[1].scatter(sub['mean_ld1'], [j]*len(sub), c=colours_fam[name],
                     s=120, zorder=5, edgecolors='white', linewidths=0.8)
axes[1].set_yticks(range(len(FAMILY)))
axes[1].set_yticklabels(FAMILY, fontsize=10)
axes[1].set_xlabel('LD1 (trajectory means)', fontsize=10)
axes[1].set_title('LDA Replica-Level Separation', fontsize=11, fontweight='bold')
axes[1].axvline(0, color='grey', lw=0.8, ls='--')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'lda_q2022p_family.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('\nLDA Summary:')
print(df_means.groupby('name')['mean_ld1'].describe().round(5))


## Section 7 — Same-Position Contrast Pairs

In [ ]:
# Define contrast pairs for direct comparison
CONTRAST_PAIRS = [
    ('L1982F', 'L1982V',  'αC-helix — L1982 series'),
    ('S1986F', 'S1986Y',  'αC-helix — S1986 series'),
    ('F2004C', 'F2004V',  'Gatekeeper-adj — F2004 series (C vs V)'),
    ('F2004L', 'F2004V',  'Gatekeeper-adj — F2004 series (L vs V)'),
    ('G2032K', 'G2032R',  'Hinge — G2032 series'),
    ('D2113G', 'D2113N',  'A-loop — D2113 series'),
    ('WT',     'Q2022P',  'Hinge — WT vs Q2022P'),
]

n_pairs = len(CONTRAST_PAIRS)
fig, axes = plt.subplots(n_pairs, 2, figsize=(10, n_pairs * 3), dpi=110)

for row, (name_a, name_b, title) in enumerate(CONTRAST_PAIRS):
    idx_a = LABELS.index(name_a)
    idx_b = LABELS.index(name_b)
    mask_a = owner == idx_a
    mask_b = owner == idx_b

    ca = PALETTE.get(name_a, '#1976D2')
    cb = PALETTE.get(name_b, '#E53935')

    for col, (mask, name, colour) in enumerate([
        (mask_a, name_a, ca), (mask_b, name_b, cb)
    ]):
        G, xc, yc = compute_fel(pc1_all[mask], pc2_all[mask])
        XX, YY = np.meshgrid(xc, yc)
        ax = axes[row][col]
        ax.pcolormesh(XX, YY, G, cmap='viridis_r', vmin=0, vmax=2.5,
                       shading='auto', rasterized=True)
        ax.set_xlim(XLIM); ax.set_ylim(YLIM)
        ax.set_title(f'{name}', fontsize=9, fontweight='bold', color=colour)
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.set_ylabel(title, fontsize=7.5, labelpad=3)

plt.suptitle('Same-Position Contrast Pairs — actout_ctlfit FEL',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'contrast_pairs_FEL.png'),
            dpi=120, bbox_inches='tight')
plt.show()


## Section 8 — Grey Highlight Plots (All 39 Systems)

In [ ]:
# ── Check if pre-generated highlight plots exist, else generate on the fly ────
highlight_dir = os.path.join(FIG_DIR, 'grey_highlight')
generated_files = glob.glob(os.path.join(highlight_dir, '*_highlight.png'))

if len(generated_files) >= 39:
    print(f'Found {len(generated_files)} pre-generated highlight plots.')
    print('Displaying summary panel...')
    from PIL import Image
    panel_path = os.path.join(highlight_dir, 'panel_all_highlight.png')
    if os.path.exists(panel_path):
        img = Image.open(panel_path)
        plt.figure(figsize=(18, 22))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Grey Highlight Plots — All 39 Systems', fontsize=14, fontweight='bold')
        plt.show()
else:
    print(f'Only {len(generated_files)} highlight plots found — generating inline...')
    os.makedirs(highlight_dir, exist_ok=True)
    n_cols_h, n_rows_h = 6, 7
    fig_h, axes_h = plt.subplots(n_rows_h, n_cols_h,
                                   figsize=(n_cols_h*3, n_rows_h*3), dpi=100)
    axes_h = axes_h.flatten()
    for i, name in enumerate(LABELS):
        ax = axes_h[i]
        mask_g = owner != i
        mask_m = owner == i
        colour = PALETTE.get(name, DEFAULT_COLOUR)
        ax.scatter(pc1_all[mask_g], pc2_all[mask_g],
                    c='#CCCCCC', s=0.4, alpha=0.2, rasterized=True)
        ax.scatter(pc1_all[mask_m], pc2_all[mask_m],
                    c=colour, s=0.9, alpha=0.8, rasterized=True)
        ax.set_xlim(XLIM); ax.set_ylim(YLIM)
        ax.set_title(name, fontsize=7, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
    for j in range(N_SYS, len(axes_h)):
        axes_h[j].axis('off')
    plt.suptitle('Grey Highlight: All 39 Systems', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(highlight_dir, 'panel_all_highlight.png'),
                dpi=100, bbox_inches='tight')
    plt.show()
    print('Panel saved.')


## Section 9 — CTL FEL Overview

In [ ]:
ctl_scores_file = os.path.join(BASE_DIR, 'ctl_scores.npy')

if os.path.exists(ctl_scores_file):
    ctl_scores = np.load(ctl_scores_file)
    ctl_pc1, ctl_pc2 = ctl_scores[:,0], ctl_scores[:,1]
    CTL_XLIM = (np.percentile(ctl_pc1,0.5), np.percentile(ctl_pc1,99.5))
    CTL_YLIM = (np.percentile(ctl_pc2,0.5), np.percentile(ctl_pc2,99.5))

    # Bimodal CTL landscape summary
    bimodal = ['WT','C2060G','D1988N','E2020K','E2131Q','F2004V','F2075V',
                'G2032K','G2032R','L1947R','L1951R','Q2022P_S1986Y','S1986Y']
    unimodal = ['D2113G','D2113N','E1935G','E1990G','F2004C','F2004L',
                 'G1971E','G2101A','L2010M','S1986F']

    print(f'CTL scores shape: {ctl_scores.shape}')
    print(f'\nBimodal CTL variants ({len(bimodal)}): {", ".join(bimodal)}')
    print(f'Unimodal CTL variants ({len(unimodal)}): {", ".join(unimodal)}')

    # Plot key CTL FEL examples side by side
    examples = ['WT','S1986F','Q2022P_S1986Y','E2131Q','G2032K','G2032R']
    fig, axes = plt.subplots(2, 3, figsize=(12, 8), dpi=110)
    axes = axes.flatten()
    for k, name in enumerate(examples):
        idx = LABELS.index(name)
        mask = owner == idx
        plot_fel(axes[k], ctl_pc1[mask], ctl_pc2[mask],
                  title=name, xlim=CTL_XLIM, ylim=CTL_YLIM)
    plt.suptitle('CTL PCA Free Energy Landscapes — Key Examples',
                  fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'CTL_FEL_examples.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'CTL scores not found at:\n  {ctl_scores_file}')
    print('Load CTL FEL figures from step3c output directory instead.')


## Section 10 — Docking Results (Q2022P Family)

In [ ]:
# Load docking results once pipeline is complete
docking_csv = os.path.join(BASE_DIR, 'docking', 'vina_results_ranked.csv')

if os.path.exists(docking_csv):
    df_dock = pd.read_csv(docking_csv)
    print(f'Docking results loaded: {df_dock.shape}')
    print(df_dock.head(20))

    # Plot binding affinity comparison
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=120)
    for ax, ligand in zip(axes, ['crizotinib', 'lorlatinib']):
        sub = df_dock[df_dock['ligand'] == ligand].copy()
        sub = sub.sort_values('best_affinity_kcal_mol')
        colours_d = [PALETTE.get(n, DEFAULT_COLOUR) for n in sub['system']]
        ax.barh(range(len(sub)), sub['best_affinity_kcal_mol'],
                 color=colours_d, edgecolor='white', linewidth=0.3)
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(sub['system'], fontsize=8)
        ax.set_xlabel('Binding Affinity (kcal/mol)', fontsize=10)
        ax.set_title(f'{ligand.capitalize()} — Vina Affinity',
                      fontsize=11, fontweight='bold')
        ax.axvline(sub[sub['system']=='WT']['best_affinity_kcal_mol'].values[0],
                    color='#2196F3', lw=1.5, ls='--', label='WT')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'docking_affinity_comparison.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Docking results not yet available.')
    print(f'Expected at: {docking_csv}')
    print('Run pipeline steps 05a → 05aa → 05ab → 05b → 05d → 05c first.')


## Section 11 — Export Summary Table

In [ ]:
# Build comprehensive per-mutant summary table
summary = pd.DataFrame({'Variant': LABELS})
summary['Owner_Index'] = range(N_SYS)
summary['N_Frames'] = [int((owner==i).sum()) for i in range(N_SYS)]

# Per-mutant PC1/PC2 means in actout_ctlfit space
summary['actout_PC1_mean'] = [round(scores[owner==i,0].mean(),4) for i in range(N_SYS)]
summary['actout_PC2_mean'] = [round(scores[owner==i,1].mean(),4) for i in range(N_SYS)]

# DBSCAN noise fractions (confirmed values)
noise_dict = dict(zip(
    ['L1982V','G2048A_NC','L1951R','F2004C','Q2022P','S1986Y','E2131Q',
     'V2098I','L2086F','E2020K','L2155S','Q2022P_S1986Y','L1947R',
     'H1999Q_NC','Q2022P_S1986F','L1982F','C2060G','F2004L','L2053V_NC',
     'D1988N','V2089M','R2078K','G2101A','WT','E1990G','L2010M',
     'D2033N','G2032R','F2075V','F2004V','E1935G','G1957A','S1986F',
     'G2032K','G1971E','D2113G','D2113N'],
    [7.27,5.21,5.18,5.14,4.51,4.31,4.28,
     3.61,3.50,2.50,2.26,2.07,1.95,
     1.64,1.53,1.45,1.27,1.26,1.36,
     1.24,1.20,0.95,0.84,0.83,0.83,0.82,
     0.79,0.68,0.55,0.44,0.34,0.30,0.25,
     0.21,0.10,0.16,0.08]
))
summary['DBSCAN_Noise_Pct'] = summary['Variant'].map(noise_dict)

# Classification
nc_variants = {'G2048A_NC','H1999Q_NC','L2053V_NC'}
compound    = {'Q2022P_S1986F','Q2022P_S1986Y'}
def classify(name):
    if name == 'WT': return 'Reference'
    if name in nc_variants: return 'Negative control'
    if name in compound: return 'Compound/suppressor'
    return 'Resistance mutant'
summary['Classification'] = summary['Variant'].apply(classify)

# If metrics available, merge pocket openness
if 'Pocket_Open_Pct' in df_metrics.columns:
    summary = summary.merge(
        df_metrics[['Variant','Pocket_Open_Pct','DFG_in_Pct']],
        on='Variant', how='left'
    )

print(summary.to_string(index=False))

# Save as CSV
out_csv = os.path.join(BASE_DIR, 'ROS1_variant_summary_table.csv')
summary.to_csv(out_csv, index=False)
print(f'\nSaved to: {out_csv}')
